# CUDA C++ Study Manual
## Matrix Addition, Matrix Multiplication & Tic-Tac-Toe

### Learning objectives
- Understand CUDA kernels, blocks, grids and thread indices.
- Implement parallel matrix addition.
- Implement parallel matrix multiplication.
- Understand CPU ↔ GPU memory transfer.
- Apply CUDA parallelism to Tic-Tac-Toe winner checking.
- Understand why these examples are useful for learning CUDA.

# 1. CUDA Programming Model

A CUDA program normally follows:

```text
CPU
 ↓
Allocate GPU memory
 ↓
Copy data CPU → GPU
 ↓
Launch CUDA kernel
 ↓
GPU executes many threads
 ↓
Copy result GPU → CPU
 ↓
Free GPU memory
```

A kernel is declared using `__global__` and launched with:

```cpp
kernel<<<grid, block>>>(arguments);
```

# 2. Matrix Addition

For:

`C = A + B`

each element is independent:

`C[i][j] = A[i][j] + B[i][j]`

Therefore, one CUDA thread can calculate one output element.

```text
Thread (0,0) → C[0][0]
Thread (0,1) → C[0][1]
Thread (0,2) → C[0][2]
...
```

In [ ]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 3

__global__ void matrixAdd(int *A, int *B, int *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int index = row * N + col;
        C[index] = A[index] + B[index];
    }
}

int main()
{
    int A[N][N] = {{1,2,3},{4,5,6},{7,8,9}};
    int B[N][N] = {{9,8,7},{6,5,4},{3,2,1}};
    int C[N][N];

    int *d_A, *d_B, *d_C;
    size_t size = N * N * sizeof(int);

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16,16);

    dim3 blocksPerGrid(
        (N + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (N + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    matrixAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, size, cudaMemcpyDeviceToHost);

    printf("Matrix Addition: C = A + B\n");

    for(int i=0;i<N;i++)
    {
        for(int j=0;j<N;j++)
            printf("%d ", C[i][j]);
        printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


In [ ]:
!nvcc matrix_add.cu -o matrix_add
!./matrix_add

Expected output:

```text
Matrix Addition: C = A + B

10 10 10
10 10 10
10 10 10
```

### Important indexing formula

```cpp
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
```

This maps a CUDA thread to a matrix position.

# 3. Matrix Multiplication

For:

`C = A × B`

each output element is:

`C[i][j] = Σ A[i][k] × B[k][j]`

One CUDA thread calculates one output element, but each thread performs a dot product using a loop.

In [ ]:
%%writefile matrix_mul.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 2

__global__ void matrixMul(int *A, int *B, int *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;

        for(int k=0;k<N;k++)
        {
            sum += A[row*N+k] * B[k*N+col];
        }

        C[row*N+col] = sum;
    }
}

int main()
{
    int A[N][N] = {{1,2},{3,4}};
    int B[N][N] = {{5,6},{7,8}};
    int C[N][N];

    int *d_A,*d_B,*d_C;
    size_t size = N*N*sizeof(int);

    cudaMalloc((void**)&d_A,size);
    cudaMalloc((void**)&d_B,size);
    cudaMalloc((void**)&d_C,size);

    cudaMemcpy(d_A,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,size,cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16,16);

    dim3 blocksPerGrid(
        (N + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (N + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    matrixMul<<<blocksPerGrid,threadsPerBlock>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(C,d_C,size,cudaMemcpyDeviceToHost);

    printf("Matrix Multiplication: C = A * B\n");

    for(int i=0;i<N;i++)
    {
        for(int j=0;j<N;j++)
            printf("%d ",C[i][j]);
        printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


In [ ]:
!nvcc matrix_mul.cu -o matrix_mul
!./matrix_mul

Expected output:

```text
Matrix Multiplication: C = A * B

19 22
43 50
```

For example:

`C[0][0] = (1×5) + (2×7) = 19`

# 4. Matrix Addition vs Matrix Multiplication

| Feature | Addition | Multiplication |
|---|---|---|
| Formula | `C[i][j]=A[i][j]+B[i][j]` | `C[i][j]=ΣA[i][k]B[k][j]` |
| Thread mapping | One thread → one output | One thread → one output |
| Work per output | One addition | Multiple multiply-add operations |
| Parallelism | Very high | Very high |
| Main concept | Element-wise parallelism | Parallel dot products |

Matrix multiplication is more computationally intensive and is therefore a major CUDA optimization example.

# 5. Tic-Tac-Toe with CUDA

The Tic-Tac-Toe board has 8 possible winning lines:

```text
Rows:
0 1 2
3 4 5
6 7 8

Columns:
0 3 6
1 4 7
2 5 8

Diagonals:
0 4 8
2 4 6
```

The CUDA strategy is:

```text
Thread 0 → check line 0
Thread 1 → check line 1
...
Thread 7 → check line 7
```

The CPU handles user input and the game loop. The GPU checks the winning lines in parallel.

In [ ]:
%%writefile tictactoe.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void checkWinner(char *board, int *results)
{
    int line = blockIdx.x * blockDim.x + threadIdx.x;

    if(line >= 8) return;

    int winningLines[8][3] = {
        {0,1,2}, {3,4,5}, {6,7,8},
        {0,3,6}, {1,4,7}, {2,5,8},
        {0,4,8}, {2,4,6}
    };

    int a = winningLines[line][0];
    int b = winningLines[line][1];
    int c = winningLines[line][2];

    if(board[a] != ' ' &&
       board[a] == board[b] &&
       board[b] == board[c])
    {
        results[line] = (board[a] == 'X') ? 1 : 2;
    }
}

void displayBoard(char board[])
{
    printf("\n %c | %c | %c\n",board[0],board[1],board[2]);
    printf("---+---+---\n");
    printf(" %c | %c | %c\n",board[3],board[4],board[5]);
    printf("---+---+---\n");
    printf(" %c | %c | %c\n\n",board[6],board[7],board[8]);
}

bool boardFull(char board[])
{
    for(int i=0;i<9;i++)
        if(board[i]==' ') return false;

    return true;
}

int main()
{
    char board[9] = {
        ' ',' ',' ',
        ' ',' ',' ',
        ' ',' ',' '
    };

    char *d_board;
    int *d_results;
    int results[8];

    cudaMalloc((void**)&d_board,9*sizeof(char));
    cudaMalloc((void**)&d_results,8*sizeof(int));

    char player='X';
    int position;

    printf("CUDA TIC-TAC-TOE\n");
    printf("Positions: 1 2 3 / 4 5 6 / 7 8 9\n");

    while(true)
    {
        displayBoard(board);

        printf("Player %c, enter position: ",player);
        scanf("%d",&position);
        position--;

        if(position<0 || position>8 || board[position]!=' ')
        {
            printf("Invalid or occupied position.\n");
            continue;
        }

        board[position]=player;

        cudaMemcpy(
            d_board,board,9*sizeof(char),
            cudaMemcpyHostToDevice
        );

        cudaMemset(d_results,0,8*sizeof(int));

        checkWinner<<<1,8>>>(d_board,d_results);
        cudaDeviceSynchronize();

        cudaMemcpy(
            results,d_results,8*sizeof(int),
            cudaMemcpyDeviceToHost
        );

        bool winner=false;

        for(int i=0;i<8;i++)
        {
            if(results[i]==1)
            {
                displayBoard(board);
                printf("Player X wins!\n");
                winner=true;
                break;
            }

            if(results[i]==2)
            {
                displayBoard(board);
                printf("Player O wins!\n");
                winner=true;
                break;
            }
        }

        if(winner) break;

        if(boardFull(board))
        {
            displayBoard(board);
            printf("Game Draw!\n");
            break;
        }

        player = (player=='X') ? 'O' : 'X';
    }

    cudaFree(d_board);
    cudaFree(d_results);

    return 0;
}


In [ ]:
!nvcc tictactoe.cu -o tictactoe
!./tictactoe

# 6. Key CUDA Concepts to Remember

### Matrix addition

```text
One thread → one matrix element
```

### Matrix multiplication

```text
One thread → one output element
             ↓
          dot product
```

### Tic-Tac-Toe

```text
One thread → one winning line
```

The common pattern is:

```text
Independent work
      ↓
CUDA threads
      ↓
Parallel execution
```

### Recommended next topics

1. Shared memory
2. Tiled matrix multiplication
3. Coalesced memory access
4. Warp execution
5. Thread divergence
6. CUDA streams
7. Unified memory
8. Atomic operations
9. cuBLAS
10. CPU vs GPU performance measurement

# 7. Exercises

### Exercise 1
Change matrix addition from `3 × 3` to `1024 × 1024`.

### Exercise 2
Modify matrix multiplication to use `4 × 4` matrices.

### Exercise 3
Print each CUDA thread's `(row, col)` for matrix addition.

### Exercise 4
Add CUDA error checking after kernel execution.

### Exercise 5
Implement matrix subtraction.

### Exercise 6
Implement matrix transpose.

### Exercise 7
Optimize matrix multiplication using shared memory.

### Exercise 8
Create a Tic-Tac-Toe AI that evaluates possible moves using CUDA threads.

### Exercise 9
Measure CPU and GPU execution time for matrix multiplication.

### Exercise 10
Compare a simple CUDA matrix multiplication with cuBLAS.